# Train the transformer on Kaggle

Builds `wav-transformer` from GitHub and trains the RM (or Vec) transformer on XQuAD (de). The trained model ends up
in the **Output** of this notebook, ready to download and to continue in the next session.

**Once**
1. Create a Kaggle account and verify your phone number (*Settings → Phone verification*); without it notebooks have no internet.
2. *Create → New Notebook*, then *File → Import Notebook* and upload this file.
3. In the right panel (*Session options*): **Internet on**, Accelerator **GPU T4 x2** (or P100) together with `USE_GPU = True`.
   At model width 128 the GPU is clearly faster (CPU: ~17 min per epoch on 4 cores). Without a GPU set Accelerator **None**
   and `USE_GPU = False`.

**Every run**
1. Adjust the settings in the first code cell.
2. *Save Version → Save & Run All (Commit)*. The run continues in the background (you can close the browser) for up to 12 hours;
   `MAX_HOURS` stops the training before that and keeps the best model.
3. When it is done: open the version → **Output** → download `model_rm.zip` (or use it as input for the next run).

**Continue in the next session**
*Add Input → Your Work → Notebooks* → choose this notebook (its last output). With `RESUME = True` the training continues that model.

Models of different width (e.g. 16 before, 128 now) cannot be continued: start with `RESUME = False` or without input.

**Use the model on your computer**: unpack `model_rm.zip` into the `STORAGE` folder of the project (it contains the model file
`transformer_rm_v1` and `transformer_rm_v1.split.json`).

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────────
REPO = "https://github.com/RodionKovalenko/rustCrates"
BRANCH = "feature/neural-network-rust"
MODEL = "rm"            # "rm" (RM transformer) or "vec" (Vec transformer)
EPOCHS = 40
BATCH_SIZE = 16
LEARNING_RATE = 0.0003
RESUME = True           # continue the model found in the notebook input; False starts a new model
USE_GPU = True          # build with --features cuda (needs Accelerator GPU); False for Accelerator None
MAX_HOURS = 11.0        # Kaggle ends a session after 12 h: stop before, the best model stays saved

INPUT_ROOT = "/kaggle/input"      # previous outputs added with "Add Input"
OUTPUT_ROOT = "/kaggle/working"   # everything here becomes the notebook output
SOURCE_DIR = "/tmp/rustCrates"    # checkout and build outside the output (the build is several GB)

In [ ]:
import glob, os, shutil, subprocess, sys, time, zipfile

MODEL_DIR = {"rm": "transformer_rm_v1", "vec": "transformer_01.04.2025"}[MODEL]
COMMAND = {"rm": "train", "vec": "train-vec"}[MODEL]

def copy_model(source, target):
    # The model is a file (older builds: a sled folder); copy either.
    if os.path.isdir(source):
        shutil.rmtree(target, ignore_errors=True)
        shutil.copytree(source, target)
    else:
        shutil.copy(source, target)

def run(cmd, cwd=None, env=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env)

# Rust (the repository's rust-toolchain file selects the version on the first cargo call)
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]

# Source
if os.path.exists(SOURCE_DIR):
    run(f"git fetch --depth 1 origin {BRANCH} && git checkout -B {BRANCH} FETCH_HEAD", cwd=SOURCE_DIR)
else:
    run(f"git clone --depth 1 --branch {BRANCH} {REPO} {SOURCE_DIR}")
run("git log -1 --oneline", cwd=SOURCE_DIR)

In [ ]:
def find_previous_model(input_root, model_dir):
    # The model from an earlier output, or a model zip to unpack. Returns the folder that contains model_dir.
    for path in sorted(glob.glob(os.path.join(input_root, "**", model_dir), recursive=True)):
        return os.path.dirname(path)
    for archive in sorted(glob.glob(os.path.join(input_root, "**", f"model_{MODEL}.zip"), recursive=True)):
        target = "/tmp/previous_model"
        shutil.rmtree(target, ignore_errors=True)
        with zipfile.ZipFile(archive) as z:
            z.extractall(target)
        if os.path.exists(os.path.join(target, model_dir)):
            return target
    return None

storage = os.path.join(SOURCE_DIR, "STORAGE")
os.makedirs(storage, exist_ok=True)
previous = find_previous_model(INPUT_ROOT, MODEL_DIR) if RESUME else None
if previous:
    copy_model(os.path.join(previous, MODEL_DIR), os.path.join(storage, MODEL_DIR))
    split = os.path.join(previous, MODEL_DIR + ".split.json")
    if os.path.exists(split):
        shutil.copy(split, storage)
    print("Continuing the model from", previous)
else:
    print("Training a new model" + (" (no previous model in the input)" if RESUME else ""))
START = "resume" if previous else "new"

In [ ]:
# Build (~10 minutes)
features = "--features cuda" if USE_GPU else ""
run(f"cargo build --release {features}", cwd=SOURCE_DIR)

In [ ]:
# Train; the full log is kept in the output as train.log
env = dict(os.environ, TRAIN_MAX_HOURS=str(MAX_HOURS))
binary = os.path.join(os.environ.get("CARGO_TARGET_DIR", os.path.join(SOURCE_DIR, "target")), "release", "wav-transformer")
cmd = [binary, COMMAND, str(EPOCHS), "all", str(BATCH_SIZE), START, str(LEARNING_RATE)]
print("$", " ".join(cmd), flush=True)
started = time.time()
with open(os.path.join(OUTPUT_ROOT, "train.log"), "w") as log:
    proc = subprocess.Popen(cmd, cwd=SOURCE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        log.flush()
        if any(key in line for key in ("Loaded", "Created", "Epoch", "LOSS", "best", "improvement", "Learning rate", "selected", "stop", "Stop", "error", "Error", "panicked", "done")):
            print(f"[{(time.time() - started) / 3600:5.2f} h] {line.rstrip()}", flush=True)
    proc.wait()
print("exit code", proc.returncode)

In [ ]:
# Output: the model, its training/validation split and one zip of both
output = os.path.join(OUTPUT_ROOT, "model")
shutil.rmtree(output, ignore_errors=True)
os.makedirs(output)
copy_model(os.path.join(storage, MODEL_DIR), os.path.join(output, MODEL_DIR))
split = os.path.join(storage, MODEL_DIR + ".split.json")
if os.path.exists(split):
    shutil.copy(split, output)
archive = shutil.make_archive(os.path.join(OUTPUT_ROOT, f"model_{MODEL}"), "zip", output)
print("Saved", archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")